# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections  in order  — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [25]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", bool(hf_token))

HF_TOKEN loaded: True


In [26]:
import duckdb
import pandas as pd

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
    """
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connection ready.")

DuckDB connection ready.


In [27]:
march_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print(march_path)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


In [28]:
test = con.sql(
    f"""
    SELECT *
    FROM read_parquet('{march_path}')
    LIMIT 5
    """
)

test

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [29]:
columns = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
    """
).df()

columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:


One row means:  One content item for one client on one report date.

 Table used:  fact_content_daily_performance

 Development window:  March 2026 (`2026-03-01` through `2026-03-31`).

 What I will predict/rank:  I will use observed search-performance signals to support ranking content items by search-performance opportunity. For later supervised modeling, the target must be defined from a future observed outcome rather than from the same-period feature values.

 Deliberately excluded:  I exclude the final June 2026 month from feature development because it is the final outcome window and could expose future information during experimentation.


In [30]:
query_1 = con.sql(
    f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{march_path}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    """
)

query_1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

In [31]:
query_2 = con.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{march_path}')
    """
)

query_2

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [32]:
availability_columns = columns[
    columns["column_name"]
    .str.contains("available", case=False, na=False)
]

availability_columns

,column_name,column_type,null,key,default,extra
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None


In [34]:
query_3 = con.sql(
    f"""
    SELECT
        COUNT(*) AS available_rows
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    """
)

query_3

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
 Features — maximum five

1.  Historical impressions  — knowable at the decision moment because they are observed search impressions from the available feature window.
2.  Historical clicks  — knowable at the decision moment because they are observed search clicks from the available feature window.
3.  Average position  — knowable at the decision moment because the search position is measured in the available observation window.
4.  Content age  — knowable at the decision moment because the content creation date is already known.
5.  Days since last update  — knowable at the decision moment because the latest update date is already known.

 Label / proxy:  A future observed search outcome is the appropriate target for a later predictive model. I will not use the same-period `trend_direction` as a legitimate future label without explicitly treating it as a proxy.

 Context:  `client_hash_id` and `content_hash_id` are used for grouping, joining, and validation, not as predictive features.

 Excluded:  `trend_direction` and `trend_pct` are excluded because the warehouse documentation identifies them as label-source fields; using them as features would leak the outcome into the model.


In [35]:
columns[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [37]:
feature_frame = con.sql(
    f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    LIMIT 10000
    """
).df()

feature_frame.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727


In [39]:
features_only = feature_frame[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
]

features_only.head()

,gsc_impressions,gsc_clicks,gsc_avg_position
0,20,0,3.350000
1,1,0,0.000000
2,125,1,4.928000
3,7,0,4.000000
4,11,0,2.272727


In [41]:
leak_frame = con.sql(
    f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    LIMIT 10000
    """
).df()

leak_frame.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727


In [42]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Using available columns in leak_frame
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

# Filter data and drop missing values
model_data = leak_frame[features].dropna()

X = model_data[features]

# Example: Predicting clicks from other features as a placeholder for a label
y = model_data["gsc_clicks"]
X_feature_set = X.drop(columns=["gsc_clicks"])

X_train, X_test, y_train, y_test = train_test_split(
    X_feature_set,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

score = model.score(X_test, y_test)

print(f"Model R^2 Score using available GSC features: {score:.4f}")

Model R^2 Score using available GSC features: 0.3664


In [43]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Using verified columns from leak_frame
honest_data = leak_frame[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
].dropna()

# Define features (X) and target (y)
X = honest_data[["gsc_impressions", "gsc_avg_position"]]
y = honest_data["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

honest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_mse = mean_squared_error(y_test, honest_pred)

print(f"Model R^2 Score: {honest_model.score(X_test, y_test):.4f}")
print(f"Mean Squared Error: {honest_mse:.4f}")

Model R^2 Score: 0.3664
Mean Squared Error: 0.5700


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
Limitation: The warehouse is an unbalanced panel: different clients have different amounts of historical data. Therefore, the same calendar month does not necessarily represent the same amount of historical context for every client.
The warehouse also contains rows where GA4 data is unavailable or NULL, so missing analytics data must not automatically be interpreted as zero engagement.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.